# RV method history

Use a live query to the NASA Exoplanet Archive at IPAC using `astroquery` to create a plot of RV semiamplitude vs. discovery date, and 
planet mass vs. discovery date.  These allow us to make updated versions of plots that used to be available from the defunct exoplanet
database and are not readily extracted from the new NASA Exoplanet Database with their plotting tools.

In [ ]:
%matplotlib inline

import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, LogLocator, NullFormatter

# astroquery for the NASA Exoplanet Archive

from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

# suppress nuisance warnings

import warnings
warnings.filterwarnings('ignore',category=UserWarning, append=True)
warnings.filterwarnings('ignore',category=RuntimeWarning, append=True)

## Standard Plot Format

Setup the standard plotting format and make the plot.  

In [ ]:
# graphic aspect ratio = width/height

aspect = 4.0/3.0 # 4:3 letter
#aspect = 16.0/9.0 # wide-screen

#
# Don't change these unless you really need to (we never have)
#
# fPage is the horizontal fraction of the page occupied by the figure, default 1.0
#
# scaleFac is the LaTeX includegraphics scaling in units of \textwidth, default 1.0
#

fPage = 1.0
scaleFac = 0.85

# Text width in inches - don't change, this is defined by the print layout

textWidth = 6.0 # inches

figFmt = 'png'
dpi = 600
plotWidth = dpi*fPage*textWidth
plotHeight = plotWidth/aspect
axisFontSize = 10
labelFontSize = 8
lwidth = 0.5
axisPad = 5
wInches = fPage*textWidth # float(plotWidth)/float(dpi)
hInches = wInches/aspect  # float(plotHeight)/float(dpi)
    
# LaTeX is used throughout for markup of symbols, Times-Roman serif font

plt.rc('text', usetex=True)
plt.rc('font', **{'family':'serif','serif':['Times-Roman'],'weight':'bold','size':'16'})

# Font and line weight defaults for axes

matplotlib.rc('axes',linewidth=lwidth)
matplotlib.rcParams.update({'font.size':axisFontSize})

# axis and label padding

plt.rcParams['xtick.major.pad']=f'{axisPad}'
plt.rcParams['ytick.major.pad']=f'{axisPad}'
plt.rcParams['axes.labelpad'] = f'{axisPad}'

## Retrieve the data

Use `astroquery` to read the NASA Exoplanet Archive selecting out all data where `discoverymethod` is 
`Radial Velocity` and return the date of discovery (`disc_year`), the RV semiamplitude (`pl_rvamp`) in 
units of meters/second, and the best estimate of the mass in Earth masses (`pl_masse`).

In [ ]:
exoData = NasaExoplanetArchive.query_criteria(table="pscomppars",
                                              where="discoverymethod like 'Radial Velocity'",
                                              select="disc_year,pl_rvamp,pl_bmasse")

discYear = np.array(exoData['disc_year'])
rvSemi = np.array(exoData['pl_rvamp'])
plMass = np.array(exoData['pl_bmasse'])

dyMin = np.min(discYear) - 1.0
dyMax = np.max(discYear) + 1.0

rvMin = 0.02 # m/s
rvMax = 2000.0 # m/s

mMin = 0.1 # M_earth
mMax = 1.5e4 # M_earth

# other numbers

rvEarth = 0.09 # m/s
rvFloor = 0.60 # m/s
rv1995 = 60.0 # m/s

yrFloor = 2013

print(f"Retrieved data for {len(discYear)} RV planets.")

## Make the plots

### RV semiamplitude vs. discovery year


In [ ]:
plotFile = f"rvDiscovery_{np.max(discYear):.0f}_4x3.png"

fig,ax = plt.subplots(figsize=(wInches,hInches),dpi=dpi)

ax.tick_params('both',length=6,width=lwidth,which='major',direction='in',top='on',right='on')
ax.tick_params('both',length=3,width=lwidth,which='minor',direction='in',top='on',right='on')

# Limits

ax.set_xlim(dyMin,dyMax)
ax.xaxis.set_major_locator(MultipleLocator(5))
ax.xaxis.set_minor_locator(MultipleLocator(1))
ax.set_xlabel(r'Discovery Year',fontsize=axisFontSize)

ax.set_ylim(rvMin,rvMax)
ax.set_yscale('log')
ax.set_yticks([0.1,1,10,100,1000])
ax.set_yticklabels(['0.1','1','10','100','1000'])
ax.set_ylabel(r'RV semiamplitude [m/s]',fontsize=axisFontSize)

ax.plot(discYear,rvSemi,'o',ms=2,mfc='black',mec='None',mew=0.0,zorder=10)
ax.grid(True,which='both',lw=0.5,alpha=0.5)

ax.hlines([1.0,rvEarth],dyMin,dyMax,ls=['--','-'],colors=['black','green'],zorder=9,lw=0.75)

ax.hlines(rvFloor,yrFloor,dyMax,ls=['--'],colors=['#bb0000'],zorder=9,lw=0.75)
ax.plot([1995,yrFloor],[rv1995,rvFloor],'--',color='blue',zorder=9,lw=0.75)

# hardcopy 

plt.savefig(plotFile,bbox_inches='tight',facecolor='white')

plt.show()

### RV planet mass  vs. discovery year

In [ ]:
plotFile = f"rvMasses_{np.max(discYear):.0f}.png"

fig,ax = plt.subplots(figsize=(wInches,hInches),dpi=dpi)

ax.tick_params('both',length=6,width=lwidth,which='major',direction='in',top='on',right='on')
ax.tick_params('both',length=3,width=lwidth,which='minor',direction='in',top='on',right='on')

# Limits

ax.set_xlim(dyMin,dyMax)
ax.xaxis.set_major_locator(MultipleLocator(5))
ax.xaxis.set_minor_locator(MultipleLocator(1))
ax.set_xlabel(r'Discovery Year',fontsize=axisFontSize)

ax.set_ylim(mMin,mMax)
ax.set_yscale('log')
ax.set_yticks([0.1,1,10,100,1000])
ax.set_yticklabels(['0.1','1','10','100','1000'])
ax.set_ylabel(r'Planet Mass [M$_\textrm{Earth}$]',fontsize=axisFontSize)

ax.plot(discYear,plMass,'o',ms=2,mfc='black',mec='None',mew=0.0,zorder=10)
ax.grid(True,which='both',lw=0.5,alpha=0.3)

ax.hlines([1.0],dyMin,dyMax,ls=['-'],colors=['#bb0000'],zorder=9,lw=0.75)

# hardcopy 

plt.savefig(plotFile,bbox_inches='tight',facecolor='white')

plt.show()